# TechEngage: Social Post Engagement Analysis

**Task ZR-26-00753** — which kinds of tech-news posts get the most engagement?

Data: 546 top tech-news stories from the Hacker News public API (collected 14 Oct 2026).
Engagement = upvotes (`score`) + discussion (`descendants`/comments).
Headline statistic is the **median** (engagement is heavy-tailed).

## 1. Load the cleaned data

In [ ]:
import csv, json
from collections import defaultdict
from statistics import median, mean

rows = list(csv.DictReader(open('../data/processed/hn_stories_clean.csv')))
for r in rows:
    for k in ('score', 'comments', 'engagement_total', 'title_len', 'title_words', 'hour_utc'):
        r[k] = int(r[k])
print(f'{len(rows)} posts loaded')
print('columns:', list(rows[0].keys()))

## 2. Engagement by post type

Link posts dominate the top stories (91%). `Ask HN`/`text` show huge medians but n<=5, so we don't trust those cells.

In [ ]:
def stats(rs):
    return {'n': len(rs),
            'med_score': round(median([r['score'] for r in rs]), 1),
            'med_comments': round(median([r['comments'] for r in rs]), 1),
            'med_total': round(median([r['engagement_total'] for r in rs]), 1)}

by_type = defaultdict(list)
for r in rows: by_type[r['post_type']].append(r)
for k, v in sorted(by_type.items(), key=lambda kv: stats(kv[1])['med_total'], reverse=True):
    s = stats(v)
    print(f"{k:6s} n={s['n']:4d}  med_upvotes={s['med_score']:7.1f}  med_comments={s['med_comments']:6.1f}")

## 3. Engagement by title length

Shorter headlines win, monotonically: <=40 chars gets ~1.8x the engagement of 61-80 chars.

In [ ]:
def bucket(n):
    return '<=40' if n <= 40 else '41-60' if n <= 60 else '61-80' if n <= 80 else '81-100' if n <= 100 else '100+'

by_len = defaultdict(list)
for r in rows: by_len[bucket(r['title_len'])].append(r)
for b in ['<=40', '41-60', '61-80', '81-100', '100+']:
    if b in by_len:
        s = stats(by_len[b])
        print(f"{b:6s} n={s['n']:4d}  med_total={s['med_total']:7.1f}")

# Pearson correlation: title length vs engagement
def pearson(xs, ys):
    n = len(xs); mx, my = mean(xs), mean(ys)
    num = sum((x-mx)*(y-my) for x, y in zip(xs, ys))
    den = (sum((x-mx)**2 for x in xs) * sum((y-my)**2 for y in ys)) ** 0.5
    return round(num/den, 3) if den else 0.0
print('r(title_len, score) =', pearson([r['title_len'] for r in rows], [r['score'] for r in rows]))
print('r(title_len, comments) =', pearson([r['title_len'] for r in rows], [r['comments'] for r in rows]))

## 4. Engagement by topic

AI has the most volume (n=121) but mid-pack engagement. Security / Apple / Open Source punch above their weight.

In [ ]:
by_topic = defaultdict(list)
for r in rows: by_topic[r['topic']].append(r)
for k, v in sorted(by_topic.items(), key=lambda kv: stats(kv[1])['med_total'], reverse=True):
    s = stats(v)
    print(f"{k:12s} n={s['n']:4d}  med_total={s['med_total']:7.1f}")

## 5. Engagement by weekday and hour (directional: 8-day window)

In [ ]:
by_wd = defaultdict(list)
for r in rows: by_wd[r['weekday']].append(r)
for d in ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']:
    if d in by_wd:
        s = stats(by_wd[d])
        print(f"{d:9s} n={s['n']:4d}  med_total={s['med_total']:7.1f}")
print()
by_hr = defaultdict(list)
for r in rows: by_hr[r['hour_utc']].append(r)
best = sorted(by_hr.items(), key=lambda kv: stats(kv[1])['med_total'], reverse=True)[:5]
print('top hours (UTC):', [(h, stats(v)['med_total']) for h, v in best])
print('note: PKT = UTC+5')

## 6. Takeaways for TechEngage

1. **Keep headlines short** (<=60 chars, ideally <=40): ~1.8x engagement vs long headlines.
2. **Lead with link-style news posts** on AI, Security, Apple, Open Source: highest volume + strong engagement.
3. **Time posts for the engagement window**: mid-week, morning-afternoon UTC (afternoon-evening PKT); avoid late-night UTC.

Limitations: HN audience skews developer/US-EU; top/best sample studies *winners* only;
topic buckets are keyword-based; weekday/hour from an 8-day window is directional.